# Etapa 3: Preparación de Datos (ETL)

En este cuaderno implementamos la etapa de preparación de datos según CRISP-ML(Q).

**Pasos a seguir:**
1. Carga de los datos (`mental_health_spain_final.csv`).
2. Inspección inicial (shape, dtypes, nulos).
3. Creación de la variable objetivo (`RISK_LEVEL`) a partir de indicadores numéricos.
4. Tratamiento de nulos y duplicados.
5. Detección y tratamiento de outliers.
6. Estandarización de variables numéricas.
7. Análisis de desbalance de clases.
8. Exportación del dataset limpio.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos e Inspección Inicial

In [ ]:
try:
    df = pd.read_csv('mental_health_spain_final.csv')
    print("Datos cargados correctamente.")
except FileNotFoundError:
    print("Error: No se encontró el archivo 'mental_health_spain_final.csv'.")
    raise

print("\n--- Dimensiones ---")
print(df.shape)

print("\n--- Tipos de Datos ---")
print(df.dtypes)

print("\n--- Primeros registros ---")
display(df.head())

print("\n--- Estadísticas descriptivas ---")
display(df.describe())

### 2. Creación de la Variable Objetivo `RISK_LEVEL`
Dado que el dataset original contiene series temporales de búsqueda (Google Trends), calculamos un índice de riesgo compuesto y lo discretizamos en Bajo, Medio y Alto.

In [ ]:
features_riesgo = ['anxiety', 'depression', 'panic_attacks', 'stress']
df['risk_score'] = df[features_riesgo].mean(axis=1)

# Discretización basada en percentiles o umbrales estáticos
# Usaremos cuartiles: < 33% Bajo, 33-66% Medio, > 66% Alto
q33 = df['risk_score'].quantile(0.33)
q66 = df['risk_score'].quantile(0.66)

def categorizar_riesgo(score):
    if score <= q33: return 'Bajo'
    elif score <= q66: return 'Medio'
    else: return 'Alto'

df['RISK_LEVEL'] = df['risk_score'].apply(categorizar_riesgo)
print("Distribución inicial de RISK_LEVEL:")
print(df['RISK_LEVEL'].value_counts())

df.drop(columns=['risk_score'], inplace=True)

### 3. Tratamiento de Valores Nulos y Duplicados

In [ ]:
print("\n--- Valores Nulos por Columna ---")
print(df.isnull().sum())

# Imputación: mediana para numéricos, moda para categóricos (si hubiera)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print("\nValores nulos después de imputación:", df.isnull().sum().sum())

print("\nDuplicados antes:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Duplicados después:", df.duplicated().sum())

### 4. Detección y Tratamiento de Outliers (IQR)
Calcularemos los límites de IQR y limitaremos los valores extremos (winsorizing).

In [ ]:
def clip_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower_bound, upper_bound)
    return df

for col in num_cols:
    df = clip_outliers(df, col)
print("Outliers limitados usando método IQR.")

### 5. Transformación y Estandarización
Eliminamos `date` ya que no será usado en el modelo transversal y normalizamos las numéricas.

In [ ]:
if 'date' in df.columns:
    df.drop('date', axis=1, inplace=True)

# Codificación de categóricas (One-Hot / Label). En este dataset, no hay categóricas originales además del target.
# Actualizamos num_cols después de eliminar 'date'
num_cols = df.select_dtypes(include=[np.number]).columns

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
print("Variables estandarizadas con StandardScaler.")
display(df.head())

### 6. Exportación del Dataset Limpio

In [ ]:
df.to_csv('cleaned_data.csv', index=False)
print("Dataset limpio guardado como 'cleaned_data.csv'")